In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
from torchvision import datasets, models, transforms

import numpy as np
import matplotlib.pyplot as plt
import PIL

import math
import os
import random
from tqdm import tqdm

from types import SimpleNamespace
from scipy.fftpack import dct, idct

from art.attacks.evasion import HopSkipJump
from art.estimators.classification import PyTorchClassifier
from art.attacks.inference.membership_inference import MembershipInferenceBlackBox
from art.attacks.inference.membership_inference import LabelOnlyDecisionBoundary
from art.attacks.inference.model_inversion import MIFace
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.manifold import TSNE

import opacus
from opacus.validators import ModuleValidator
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager

In [ ]:
use_cuda = True
device = torch.device("cuda" if use_cuda else "cpu")

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion *
                               planes, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion*planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512*block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])

In [ ]:
model = ResNet18()
model.load_state_dict(torch.load('./checkpoint/final_model.pth')['net'])
model = model.to(device)

In [ ]:
import numpy as np

class_names = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

def imshow(img, mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010)):
    img = img.numpy()
    for i in range(3):
        img[i] = std[i] * img[i] + mean[i]
    img = np.transpose(img, (1, 2, 0))
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.show()

def imshow_batch(input, title, mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010)):
    input = input.numpy()
    for i in range(3):
        input[i] = std[i] * input[i] + mean[i]
    input = np.transpose(input, (1, 2, 0))
    input = np.clip(input, 0, 1)
    plt.imshow(input)
    plt.title(title)
    plt.show()


# load a batch of validation image
iterator = iter(testloader)

# visualize a batch of validation image (the first 4 images)
inputs, classes = next(iterator)
out = torchvision.utils.make_grid(inputs[5:9])
imshow_batch(out, title='original labels: ' + str([class_names[x] for x in classes[5:9]]))

In [ ]:
MEAN = [0.4914, 0.4822, 0.4465]
STD = [0.2023, 0.1994, 0.2010]


def apply_normalization(imgs):
    imgs_tensor = imgs.clone()
    if imgs.dim() == 3: # color image
        for i in range(imgs_tensor.size(0)):
            imgs_tensor[i, :, :] = (imgs_tensor[i, :, :] - MEAN[i]) / STD[i]
    else: # grayscale image
        for i in range(imgs_tensor.size(1)):
            imgs_tensor[:, i, :, :] = (imgs_tensor[:, i, :, :] - MEAN[i]) / STD[i]
    return imgs_tensor

def get_preds(model, inputs, batch_size, return_cpu=True):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_batches = int(math.ceil(inputs.size(0) / float(batch_size)))
    all_preds, all_probs = None, None
    for i in range(num_batches):
        upper = min((i + 1) * batch_size, inputs.size(0))
        input = apply_normalization(inputs[(i * batch_size):upper])
        with torch.no_grad():
            output = torch.nn.Softmax(dim=1)(model(input.to(device)))
        prob, pred = output.max(1)
        if return_cpu:
            prob = prob.cpu()
            pred = pred.cpu()
        if i == 0:
            all_probs = prob
            all_preds = pred
        else:
            all_probs = torch.cat((all_probs, prob), 0)
            all_preds = torch.cat((all_preds, pred), 0)
    return all_preds, all_probs

def sample_gaussian_torch(image_size, dct_ratio=1.0):
    x = torch.zeros(image_size)
    fill_size = int(image_size[-1] * dct_ratio)
    x[:, :, :fill_size, :fill_size] = torch.randn(x.size(0), x.size(1), fill_size, fill_size)
    if dct_ratio < 1.0:
        x = torch.from_numpy(idct(idct(x.numpy(), axis=3, norm='ortho'), axis=2, norm='ortho'))
    return x

In [ ]:
criterion = nn.CrossEntropyLoss()
#iterator = iter(testloader)
inputs, labels = next(iterator)
inputs = inputs.to(device)
labels = labels.to(device)

outputs = model(inputs)
preds, _ = get_preds(model, inputs, batch_size=8, return_cpu=False)
images = torchvision.utils.make_grid(inputs[:4])
imshow_batch(images.cpu(), title='original labels: ' + str([int(x) for x in labels[0:4]]) +
                         '\npredicted labels: ' + str([int(x) for x in preds[0:4]]))


In [ ]:
def unnormalize(img, mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010)):
    img = img.numpy()
    for i in range(3): 
        img[i] = img[i] * std[i] + mean[i]
    img = np.transpose(img, (1, 2, 0))  
    img = np.clip(img, 0, 1)  
    return img


def visualize_comparison(original_images, adversarial_images, original_labels, adversarial_labels, title):
    batch_size = original_images.size(0)
    figure, axs = plt.subplots(2, batch_size, figsize=(12, 4))
    figure.suptitle(title)
    
    for i in range(batch_size):
        original_img = unnormalize(original_images[i])
        axs[0, i].imshow(original_img, interpolation = 'none')
        axs[0, i].set_title(f"Original: {class_names[original_labels[i]]}")
        axs[0, i].axis('off')

        adversarial_img = unnormalize(adversarial_images[i])
        axs[1, i].imshow(adversarial_img, interpolation = 'none')
        axs[1, i].set_title(f"Adversarial: {class_names[adversarial_labels[i]]}")
        axs[1, i].axis('off')

    plt.show()

classifier = PyTorchClassifier(
    model=model,
    clip_values=(0, 255),
    loss=nn.CrossEntropyLoss(),
    optimizer=torch.optim.SGD(model.parameters(), lr = 0.1, momentum=0.9, weight_decay=5e-4),
    input_shape=(3, 32, 32),
    nb_classes=len(class_names),
    preprocessing=(0,1)
)

attack = HopSkipJump(classifier=classifier, targeted=False, max_iter=500)

num_samples = 4
x_samples = inputs[:num_samples].to(device)
original_labels = labels[:num_samples]

x_test_adv = []
for i in range(num_samples):
    x_adv = attack.generate(x=x_samples[i:i+1].cpu().numpy())
    x_test_adv.append(torch.tensor(x_adv))

x_test_adv = torch.cat(x_test_adv)
adversarial_preds, _ = get_preds(model.to(device), x_test_adv.to(device), batch_size=num_samples, return_cpu=True)
visualize_comparison(x_samples.cpu(), x_test_adv, original_labels, adversarial_preds, "Original vs Adversarial Images")

In [ ]:
x_test_adv = torch.cat(x_test_adv)
adversarial_preds, _ = get_preds(model.to(device), x_test_adv.to(device), batch_size=num_samples, return_cpu=True)
visualize_comparison(x_samples.cpu(), x_test_adv, original_labels, adversarial_preds, "Original vs Adversarial Images")

In [ ]:
class_names = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

classifier = PyTorchClassifier(
    model=model,
    clip_values=(0, 1),
    loss=nn.CrossEntropyLoss(),
    optimizer=torch.optim.SGD(model.parameters(), lr = 0.1, momentum=0.9, weight_decay=5e-4),
    input_shape=(3, 32, 32),
    nb_classes=len(class_names),
    preprocessing=(0,1)
)

transform_stl = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_stl = datasets.STL10(root='./data', split='train', download=True, transform=transform_stl)


indices_train = torch.randperm(len(trainset))[:100]
indices_test = torch.randperm(len(test_stl))[:100]

subset_train_dataset = torch.utils.data.Subset(trainset, indices_train)
subset_test_dataset = torch.utils.data.Subset(test_stl, indices_test)

train_loader = torch.utils.data.DataLoader(subset_train_dataset, batch_size=100, shuffle=False)
test_loader = torch.utils.data.DataLoader(subset_test_dataset, batch_size=100, shuffle=False)

x_train, y_train = next(iter(train_loader))
x_test, y_test = next(iter(test_loader))

attack = LabelOnlyDecisionBoundary(classifier)
attack.calibrate_distance_threshold_unsupervised(num_samples=400, max_queries=2, top_t=60)

inferred_train = attack.infer(x_train.numpy(), y_train.numpy())
inferred_test = attack.infer(x_test.numpy(), y_test.numpy())

y_true = np.concatenate([np.ones(len(x_train)), np.zeros(len(x_test))])
y_pred = np.concatenate([inferred_train, inferred_test])

auc_score = roc_auc_score(y_true, y_pred)
print(f"AUC score: {auc_score}")

In [ ]:
from sklearn.metrics import auc, roc_curve

fpr, tpr, thresholds = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

In [ ]:
attack = MIFace(classifier)

x_test, y_test = next(iter(testloader))
x_test = x_test.numpy()
y_test = y_test.numpy()

reconstructed_images = attack.infer(x = x_test, y = y_test)

reconstructed_images

In [ ]:
actual = []
deep_features = []

model.eval()
with torch.no_grad():
    for data in trainloader:
        images, labels = data[0].to(device), data[1].to(device)
        features = model(images)

        deep_features += features.cpu().numpy().tolist()
        actual += labels.cpu().numpy().tolist()

tsne = TSNE(n_components=2, random_state=0) 
cluster = np.array(tsne.fit_transform(np.array(deep_features)))
actual = np.array(actual)

plt.figure(figsize=(10, 10))
cifar = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
for i, label in zip(range(10), cifar):
    idx = np.where(actual == i)
    plt.scatter(cluster[idx, 0], cluster[idx, 1], marker='.', label=label)

plt.legend()
plt.show()

In [ ]:
actual = []
deep_features = []

model.eval()
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        features = model(images)

        deep_features += features.cpu().numpy().tolist()
        actual += labels.cpu().numpy().tolist()

tsne = TSNE(n_components=2, random_state=0) 
cluster = np.array(tsne.fit_transform(np.array(deep_features)))
actual = np.array(actual)

plt.figure(figsize=(10, 10))
cifar = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
for i, label in zip(range(10), cifar):
    idx = np.where(actual == i)
    plt.scatter(cluster[idx, 0], cluster[idx, 1], marker='.', label=label)

plt.legend()
plt.show()

In [ ]:
from torchvision.datasets import STL10

classifier = PyTorchClassifier(
    model=model,
    clip_values=(0, 1),
    loss=nn.CrossEntropyLoss(),
    optimizer=torch.optim.Adam(model.parameters(), lr=0.001),
    input_shape=(3, 32, 32),
    nb_classes=len(class_names)
)

attack = HopSkipJump(classifier=classifier, targeted=False, max_iter=300, max_eval=12000, init_eval=50)

transform_stl10 = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

cifar10_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
stl10_dataset = models.STL10(root='./data', split='train', download=True, transform=transform_stl10)

cifar10_loader = torch.utils.data.DataLoader(cifar10_dataset, batch_size=60, shuffle=True)
stl10_loader = torch.utils.data.DataLoader(stl10_dataset, batch_size=60, shuffle=True)

cifar_images, cifar_labels = next(iter(cifar10_loader))
stl_images, stl_labels = next(iter(stl10_loader))

cifar_images, cifar_labels = cifar_images.to(device), cifar_labels.to(device)
stl_images, stl_labels = stl_images.to(device), stl_labels.to(device)

cifar_adv = attack.generate(x=cifar_images.cpu().numpy())
stl_adv = attack.generate(x=stl_images.cpu().numpy())

cifar_adv = torch.tensor(cifar_adv).to(device)
stl_adv = torch.tensor(stl_adv).to(device)

cifar_l2_distances = torch.norm(cifar_images.view(cifar_images.size(0), -1) - cifar_adv.view(cifar_adv.size(0), -1), dim=1)
stl_l2_distances = torch.norm(stl_images.view(stl_images.size(0), -1) - stl_adv.view(stl_adv.size(0), -1), dim=1)

cifar_l2_distances_np = cifar_l2_distances.cpu().numpy()
stl_l2_distances_np = stl_l2_distances.cpu().numpy()

data = [stl_l2_distances_np,cifar_l2_distances_np]
labels = ['Train Dataset', 'Non-train Dataset']

plt.figure(figsize=(8, 6))
plt.boxplot(data, labels=labels)
plt.title('Comparison of L2 Distances between Original and Adversarial Images')
plt.ylabel('L2 Distance')
plt.grid(axis='y')
plt.show()

In [ ]:
data = [cifar_l2_distances_np,stl_l2_distances_np]
labels = ['Train Dataset', 'Non-train Dataset']

plt.figure(figsize=(10, 6))
import seaborn as sns
sns.set(style="whitegrid")

box = plt.boxplot(data, labels=labels, patch_artist=True, medianprops=dict(color="violet"), showmeans=True)

for whisker in box['whiskers']:
    whisker.set(color='blue', linewidth=1.5)

for cap in box['caps']:
    cap.set(color='green', linewidth=1.5)

for patch, color in zip(box['boxes'], ['lightblue', 'lightgreen']):
    patch.set_facecolor(color)

plt.title('Comparison of L2 Distances between Original and Adversarial Images', fontsize=14)
plt.ylabel('L2 Distance', fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()

DP Model

In [ ]:
MAX_GRAD_NORM = 1.2
DELTA = 1e-5
EPOCHS = 25

In [ ]:
model_full_10 = ResNet18()
model_full_10.load_state_dict(torch.load('./checkpoint/final_model.pth')['net'])
model_full_10 = ModuleValidator.fix(model_full_10)
model_full_10 = model_full_10.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_full_10.parameters(), lr=0.1,
                      momentum=0.9, weight_decay=5e-4)

In [ ]:
privacy_engine_full_10 = PrivacyEngine()
model_full_10, optimizer, trainloader = privacy_engine_full_10.make_private_with_epsilon(
    module = model_full_10,
    optimizer = optimizer,
    data_loader = trainloader,
    max_grad_norm = MAX_GRAD_NORM,
    epochs = 25,
    target_epsilon = 10,
    target_delta = DELTA,
)

print(f"Using sigma={optimizer.noise_multiplier} and C={MAX_GRAD_NORM}")

In [ ]:
def accuracy(preds, labels):
    return (preds == labels).mean()

def train(model, train_loader, optimizer, epoch, device, privacy_engine_input):
    DELTA = 1e-5
    model.train()
    criterion = nn.CrossEntropyLoss()

    losses = []
    top1_acc = []


    for i, (images, target) in enumerate(train_loader):
        images = images.to(device)
        target = target.to(device)
        output = model(images)
        loss = criterion(output, target)
        loss.backward(retain_graph = True)

        preds = np.argmax(output.detach().cpu().numpy(), axis=1)
        labels = target.detach().cpu().numpy()
        
        acc = accuracy(preds, labels)

        losses.append(loss.item())
        top1_acc.append(acc)

        optimizer.step()
        optimizer.zero_grad()

    epsilon = privacy_engine_input.get_epsilon(DELTA)
    print(
        f"\tTrain Epoch: {epoch} \t"
        f"Loss: {np.mean(losses):.6f} "
        f"Acc@1: {np.mean(top1_acc) * 100:.6f} "
        f"(ε = {epsilon:.2f}, δ = {DELTA})"
    )

def test(model, test_loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    losses = []
    top1_acc = []

    with torch.no_grad():
        for images, target in test_loader:
            images = images.to(device)
            target = target.to(device)

            output = model(images)
            loss = criterion(output, target)
            preds = np.argmax(output.detach().cpu().numpy(), axis=1)
            labels = target.detach().cpu().numpy()
            acc = accuracy(preds, labels)

            losses.append(loss.item())
            top1_acc.append(acc)

    top1_avg = np.mean(top1_acc)

    print(
        f"\tTest set:"
        f"Loss: {np.mean(losses):.6f} "
        f"Acc: {top1_avg * 100:.6f} "
    )
    return np.mean(top1_acc)

In [ ]:
for epoch in tqdm(range(25), desc="Epoch", unit="epoch"):
    train(model_full_10, trainloader, optimizer, epoch + 1, device, privacy_engine_full_10)

In [ ]:
top1_acc = test(model_full_10, testloader, device)

DP - Last 2 Layer

In [ ]:
model_last2_10 = ResNet18()
model_last2_10.load_state_dict(torch.load('./checkpoint/final_model.pth')['net'])
resnet_modules = list(model_last2_10.children())
model_last2_10_back = nn.Sequential(*resnet_modules[:-2])
model_last2_10_head = nn.Sequential(*resnet_modules[-2:-1], nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(512, 10))
model_last2_10_head = ModuleValidator.fix(model_last2_10_head)


model_last2_10_back.to(device)
model_last2_10_head.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_last2_10_head.parameters(), lr=0.1,
                      momentum=0.9, weight_decay=5e-4)

In [ ]:
privacy_engine_last2_10 = PrivacyEngine()
model_last2_10_head, optimizer, trainloader = privacy_engine_last2_10.make_private_with_epsilon(
    module = model_last2_10_head,
    optimizer = optimizer,
    data_loader = trainloader,
    max_grad_norm = MAX_GRAD_NORM,
    epochs = 25,
    target_epsilon = 10,
    target_delta = DELTA,
)

print(f"Using sigma={optimizer.noise_multiplier} and C={MAX_GRAD_NORM}")

In [ ]:
def accuracy(preds, labels):
    return (preds == labels).mean()

def train_sep(back, head, train_loader, optimizer, epoch, device, privacy_engine_input):
    DELTA = 1e-5
    back.eval()
    head.train()
    criterion = nn.CrossEntropyLoss()

    losses = []
    top1_acc = []


    for i, (images, target) in enumerate(train_loader):
        images = images.to(device)
        target = target.to(device)

        with torch.no_grad():
            images = back(images)

        output = head(images)
        loss = criterion(output, target)
        loss.backward(retain_graph = True)

        preds = np.argmax(output.detach().cpu().numpy(), axis=1)
        labels = target.detach().cpu().numpy()

        acc = accuracy(preds, labels)

        losses.append(loss.item())
        top1_acc.append(acc)

        optimizer.step()
        optimizer.zero_grad()

    epsilon = privacy_engine_input.get_epsilon(DELTA)
    print(
        f"\tTrain Epoch: {epoch} \t"
        f"Loss: {np.mean(losses):.6f} "
        f"Acc@1: {np.mean(top1_acc) * 100:.6f} "
        f"(ε = {epsilon:.2f}, δ = {DELTA})"
    )

def test_sep(back, head, test_loader, device):
    back.eval()
    head.eval()
    criterion = nn.CrossEntropyLoss()
    losses = []
    top1_acc = []

    with torch.no_grad():
        for images, target in test_loader:
            images = images.to(device)
            target = target.to(device)

            images = back(images)
            output = head(images)
            loss = criterion(output, target)
            preds = np.argmax(output.detach().cpu().numpy(), axis=1)
            labels = target.detach().cpu().numpy()
            acc = accuracy(preds, labels)

            losses.append(loss.item())
            top1_acc.append(acc)

    top1_avg = np.mean(top1_acc)

    print(
        f"\tTest set:"
        f"Loss: {np.mean(losses):.6f} "
        f"Acc: {top1_avg * 100:.6f} "
    )
    return np.mean(top1_acc)

In [ ]:
for epoch in tqdm(range(25), desc="Epoch", unit="epoch"):
    train_sep(model_last2_10_back, model_last2_10_head, trainloader, optimizer, epoch + 1, device, privacy_engine_last2_10)

In [ ]:
top1_acc = test_sep(model_last2_10_back, model_last2_10_head, testloader, device)